In [1]:
import pandas as pd
import duckdb

df_device_alarm_daily = pd.DataFrame({
    "device_id": [
        "R05", "R05", "R05", "R05",
        "R16", "R16", "R16", "R16",
        "R34", "R34", "R34", "R34"
    ],
    "stat_date": [
        "2026-07-01",
        "2026-07-02",
        "2026-07-03",
        "2026-07-05",
        "2026-07-01",
        "2026-07-02",
        "2026-07-04",
        "2026-07-05",
        "2026-07-01",
        "2026-07-02",
        "2026-07-03",
        "2026-07-04"
    ],
    "alarm_count": [
        2, 5, 3, 6,
        1, 1, 4, 2,
        4, 2, 2, 5
    ]
})

df_device_alarm_daily["stat_date"] = pd.to_datetime(
    df_device_alarm_daily["stat_date"]
)

df_device_alarm_daily

,device_id,stat_date,alarm_count
0,R05,2026-07-01,2
1,R05,2026-07-02,5
2,R05,2026-07-03,3
3,R05,2026-07-05,6
4,R16,2026-07-01,1
5,R16,2026-07-02,1
6,R16,2026-07-04,4
7,R16,2026-07-05,2
8,R34,2026-07-01,4
9,R34,2026-07-02,2


# SQL Daily Review：设备告警次数前后比较

## 题目背景

设备每天会统计当日产生的告警次数。

现在需要将每条记录与同一设备的上一条记录进行比较，观察告警次数是增加、减少，还是保持不变。

## 题目要求

按照 `device_id` 分组，并按照 `stat_date` 升序排列。

为每条记录计算：

1. 上一条记录的日期；
2. 上一条记录的告警次数；
3. 当前告警次数与上一条记录的差值；
4. 告警次数的变化类型。

### 输出字段

| 字段 | 含义 |
|---|---|
| `device_id` | 设备编号 |
| `stat_date` | 当前记录日期 |
| `alarm_count` | 当前告警次数 |
| `previous_date` | 同一设备上一条记录的日期 |
| `previous_alarm_count` | 同一设备上一条记录的告警次数 |
| `alarm_change` | 当前告警次数减去上一条记录告警次数 |
| `change_type` | 告警次数变化类型 |

### `change_type` 判断规则

| 条件 | `change_type` |
|---|---|
| 没有上一条记录 | `FIRST_RECORD` |
| `alarm_change > 0` | `INCREASE` |
| `alarm_change < 0` | `DECREASE` |
| `alarm_change = 0` | `NO_CHANGE` |

### 关于日期缺失

`LAG()` 获取的是同一设备的上一条记录，不一定是前一个自然日。

例如：

```text
2026-07-03
2026-07-05
```

对于 `2026-07-05`，上一条记录仍然是 `2026-07-03`。

本题不需要补齐缺失日期，也不要求上一条记录必须是前一个自然日。

### 最终排序

按照以下顺序排列：

1. `device_id` 升序；
2. `stat_date` 升序。

## 解题要求

- 使用窗口函数 `LAG()`；
- 按照 `device_id` 分区；
- 按照 `stat_date` 排序；
- 使用 CTE 分步骤完成；
- 不使用自连接；
- 不重复书写相同的 `LAG()` 表达式，先在 CTE 中生成上一条记录的信息，再进行差值和类型判断。

In [7]:
query = """
WITH previous_table AS (
    SELECT
        device_id,
        stat_date,
        alarm_count,
        LAG(stat_date) OVER (
            PARTITION BY device_id
            ORDER BY stat_date
        ) AS previous_date,
        LAG(alarm_count) OVER (
            PARTITION BY device_id
            ORDER BY stat_date
        ) AS previous_alarm_count
    FROM df_device_alarm_daily
),

comparison_table AS (
    SELECT
        device_id,
        stat_date,
        alarm_count,
        previous_date,
        previous_alarm_count,
        alarm_count - previous_alarm_count AS alarm_change
    FROM previous_table
)

SELECT
    device_id,
    stat_date,
    alarm_count,
    previous_date,
    previous_alarm_count,
    alarm_change,
    CASE
        WHEN previous_date IS NULL THEN 'FIRST_RECORD'
        WHEN alarm_change > 0 THEN 'INCREASE'
        WHEN alarm_change < 0 THEN 'DECREASE'
        WHEN alarm_change = 0 THEN 'NO_CHANGE'
    END AS change_type
FROM comparison_table
ORDER BY
    device_id,
    stat_date;
"""

df = duckdb.execute(query).fetchdf()
df

,device_id,stat_date,alarm_count,previous_date,previous_alarm_count,alarm_change,change_type
0,R05,2026-07-01,2,NaT,<NA>,<NA>,FIRST_RECORD
1,R05,2026-07-02,5,2026-07-01,2,3,INCREASE
2,R05,2026-07-03,3,2026-07-02,5,-2,DECREASE
3,R05,2026-07-05,6,2026-07-03,3,3,INCREASE
4,R16,2026-07-01,1,NaT,<NA>,<NA>,FIRST_RECORD
5,R16,2026-07-02,1,2026-07-01,1,0,NO_CHANGE
6,R16,2026-07-04,4,2026-07-02,1,3,INCREASE
7,R16,2026-07-05,2,2026-07-04,4,-2,DECREASE
8,R34,2026-07-01,4,NaT,<NA>,<NA>,FIRST_RECORD
9,R34,2026-07-02,2,2026-07-01,4,-2,DECREASE
